# 33 — Federated learning: FedAvg in one scalar

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/033-federated-setting-fedavg.ipynb)

Three phones each hold a different number of examples and each would pull the model to a different value. Federated averaging trains locally, then combines the copies on a server. You will compute that combination by hand, run six rounds, and see which clients the final model actually serves.

## 1. The numbers: three clients, one average

Each client stores its own examples, so each client has a different idea of the right model. Here the model is a single number `w`, and a client's ideal value is the average target in its own data. The server must merge those three wishes into one number.

In [ ]:
import numpy as np                                  # the array library, preinstalled on Colab

sizes = np.array([100, 200, 700])                   # examples held by clients 1, 2 and 3
targets = np.array([0.1, 0.5, 0.9])                 # the w each client would pick alone
shares = sizes / sizes.sum()                        # each client's share of all the data
fedavg = float(shares @ targets)                    # weighted average: 0.74
plain = float(targets.mean())                       # equal average: 0.50

print("examples per client:", sizes)                # [100 200 700]
print("share of the data:", shares)                 # [0.1 0.2 0.7]
print("FedAvg global w:", round(fedavg, 3))         # 0.74
print("equal-weight global w:", round(plain, 3))    # 0.50
print(f"client 3 holds {shares[2]:.0%} of the data")   # 70%

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, which Colab already has.</li><li><code>sizes = ...</code> — how many examples each of the three clients owns.</li><li><code>targets = ...</code> — the model value each client would choose on its own.</li><li><code>shares = ...</code> — divide by the total, giving each client's fraction of the data.</li><li><code>fedavg = ...</code> — the weighted mean; <code>@</code> multiplies then sums.</li><li><code>plain = ...</code> — the unweighted mean, for comparison.</li><li><code>print("examples per client"...)</code> — how lopsided the split is.</li><li><code>print("share of the data"...)</code> — the three weights the server will use.</li><li><code>print("FedAvg global w"...)</code> — the answer FedAvg proposes.</li><li><code>print("equal-weight global w"...)</code> — the answer a naive server proposes.</li><li><code>print(f"client 3 holds ...")</code> — the f-string formats a fraction as a percent.</li></ul>

</details>

## 2. Step by step: six rounds of FedAvg

A round has two halves. Every client takes a gradient step on its own data, then the server replaces all copies with their weighted average. Starting from zero, follow the number that the clients agree on.

In [ ]:
w = 0.0                                             # the server's starting model
local = np.zeros(3)                                 # one copy of the model per client
lr = 0.5                                            # size of one local gradient step
history = []                                        # global w after every round

for r in range(6):                                  # six communication rounds
    local = local - lr * (local - targets)          # each client steps toward its own target
    w = float(shares @ local)                       # server: average, weighted by data
    history.append(w)                               # remember the global model
    print(f"round {r+1}: clients {np.round(local, 2)} -> global {w:.3f}")   # readout
    local = np.full(3, w)                           # broadcast the global model back

print("final global w:", round(w, 3))               # 0.728, still climbing to 0.74

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>w = 0.0</code> — the server has no information yet, so it starts at zero.</li><li><code>local = np.zeros(3)</code> — three copies, one sent to each client.</li><li><code>lr = 0.5</code> — how far one local step moves the model.</li><li><code>history = []</code> — an empty list; we append the global value each round.</li><li><code>for r in range(6)</code> — six rounds of local work plus one server average.</li><li><code>local = local - lr * (local - targets)</code> — one gradient step for all clients at once.</li><li><code>w = float(shares @ local)</code> — the server's weighted average of the three copies.</li><li><code>history.append(w)</code> — keep the number for the plot in section 3.</li><li><code>print(f"round ...")</code> — shows the client copies, then the new global value.</li><li><code>local = np.full(3, w)</code> — every client restarts from the same averaged model.</li><li><code>print("final global w"...)</code> — the value the rounds settled on.</li></ul>

</details>

## 3. Plot: where the global model lands

The curve shows the global model after each round. The green line is the weighted mean of the three targets and the red dotted line is the equal mean; the gap between them is the cost of ignoring how much data each client holds.

In [ ]:
import matplotlib.pyplot as plt                     # plotting library, preinstalled on Colab

rounds = np.arange(1, len(history) + 1)             # x-axis: round numbers 1 to 6
plt.figure(figsize=(6, 3))                          # a wide, short figure
plt.plot(rounds, history, "o-", color="tab:blue", label="global w")        # the trajectory
plt.axhline(fedavg, color="tab:green", linestyle="--", label="weighted 0.74")   # target
plt.axhline(plain, color="tab:red", linestyle=":", label="equal 0.50")          # shortcut
plt.ylim(0, 1)                                      # room above and below the values
plt.xlabel("communication round")                   # x-axis label
plt.ylabel("global w")                              # y-axis label
plt.legend(loc="lower right")                       # name the three lines
plt.title("FedAvg climbs to the data-weighted mean")   # the takeaway
plt.show()                                          # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — loads the plotting library.</li><li><code>rounds = ...</code> — one x value per entry in <code>history</code>.</li><li><code>plt.figure(figsize=(6, 3))</code> — a wide, short canvas.</li><li><code>plt.plot(...)</code> — the global model against the round number.</li><li><code>plt.axhline(fedavg, ...)</code> — a horizontal line at the weighted mean.</li><li><code>plt.axhline(plain, ...)</code> — a second line at the equal mean.</li><li><code>plt.ylim(0, 1)</code> — both lines and the curve stay visible.</li><li><code>plt.xlabel(...)</code> — names the horizontal axis.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend(...)</code> — a key for the curve and the two lines.</li><li><code>plt.title(...)</code> — states what the reader should conclude.</li><li><code>plt.show()</code> — draws the finished figure.</li></ul>

</details>

## 4. Interactive: local epochs on a slider

One local gradient step per round is a choice, not a law. Drag the slider to give every client several steps between averages and watch the curve reach its endpoint sooner. The title reports the final global value.

In [ ]:
from ipywidgets import interact                      # sliders that rerun a function

def run(epochs=1):                                   # epochs = local steps per round
    w = 0.0                                          # server starts from zero
    local = np.zeros(3)                              # three client copies
    path = []                                        # global w per round
    for r in range(6):                               # same six rounds as before
        for e in range(epochs):                      # extra local work per round
            local = local - lr * (local - targets)   # each client moves toward its target
        w = float(shares @ local)                    # weighted server average
        local = np.full(3, w)                        # broadcast the average back
        path.append(w)                               # record this round
    plt.figure(figsize=(6, 3))                       # fresh figure on every drag
    plt.plot(range(1, 7), path, "o-", color="tab:purple")   # trajectory
    plt.axhline(fedavg, color="grey", linestyle="--")       # the weighted mean
    plt.ylim(0, 1)                                   # fixed axes, so nothing jumps
    plt.xlabel("communication round")                # x-axis label
    plt.ylabel("global w")                           # y-axis label
    plt.title(f"{epochs} local epoch(s): final w = {path[-1]:.3f}")   # live readout
    plt.show()                                       # draw it

interact(run, epochs=(1, 5, 1))                      # slider: 1 to 5 local epochs

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — Colab ships the widgets library.</li><li><code>def run(epochs=1)</code> — the function that redraws one setting.</li><li><code>w = 0.0</code> — every drag restarts the server from zero.</li><li><code>local = np.zeros(3)</code> — a fresh copy per client.</li><li><code>path = []</code> — collects this run's global values.</li><li><code>for r in range(6)</code> — the outer loop over communication rounds.</li><li><code>for e in range(epochs)</code> — the inner loop over local steps.</li><li><code>local = ...</code> — the same gradient step, repeated <code>epochs</code> times.</li><li><code>w = ...</code> — the server averages after the local work.</li><li><code>local = np.full(3, w)</code> — the broadcast that ends the round.</li><li><code>path.append(w)</code> — store the value for the line plot.</li><li><code>plt.figure(...)</code> — a new figure for each slider position.</li><li><code>plt.plot(...)</code> — the global value per round.</li><li><code>plt.axhline(...)</code> — a reference line at the weighted mean.</li><li><code>plt.ylim(0, 1)</code> — fixed axes, so nothing jumps.</li><li><code>plt.xlabel("communication round")</code> — names the horizontal axis.</li><li><code>plt.ylabel("global w")</code> — names the vertical axis.</li><li><code>plt.title(...)</code> — the live readout of epochs and final value.</li><li><code>plt.show()</code> — draw the frame.</li><li><code>interact(run, epochs=(1, 5, 1))</code> — the slider and its range.</li></ul>

</details>

## 5. When it breaks: equal weights ignore 70% of the data

Replace the weighted average with a plain mean and the 700-example client becomes a minority in its own federation. The bars compare each client's squared error under the two servers; the big client's error jumps sixfold.

In [ ]:
import numpy as np                                # reload the array library
import matplotlib.pyplot as plt                   # reload the plotting library

err_fed = (fedavg - targets) ** 2                 # squared error per client under FedAvg
err_eq = (plain - targets) ** 2                   # same, under equal weighting
loss_fed = float(shares @ err_fed)                # data-weighted total loss, FedAvg
loss_eq = float(shares @ err_eq)                  # data-weighted total loss, equal
print("weighted loss, FedAvg:", round(loss_fed, 4))    # 0.0704
print("weighted loss, equal :", round(loss_eq, 4))     # 0.1280
print("big client error, FedAvg:", round(float(err_fed[2]), 3))   # 0.026
print("big client error, equal :", round(float(err_eq[2]), 3))    # 0.160

idx = np.arange(3)                                # three x positions, one per client
plt.figure(figsize=(6, 3))                        # a wide, short figure
plt.bar(idx - 0.2, err_fed, width=0.4, label="FedAvg", color="tab:green")       # weighted
plt.bar(idx + 0.2, err_eq, width=0.4, label="equal", color="tab:red")           # unweighted
plt.xticks(idx, ["100 ex", "200 ex", "700 ex"])   # name the clients
plt.ylabel("squared error on that client")        # y-axis label
plt.legend()                                      # name the two bar groups
plt.title("equal weights blow up the 70% client") # the takeaway
plt.show()                                        # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — the array library, reloaded for this cell.</li><li><code>import matplotlib.pyplot as plt</code> — the plotting library.</li><li><code>err_fed = ...</code> — squared distance from each client's target to the FedAvg model.</li><li><code>err_eq = ...</code> — the same distances for the equally weighted model.</li><li><code>loss_fed = ...</code> — average those errors, weighting by data size.</li><li><code>loss_eq = ...</code> — the same weighted total for the equal model.</li><li><code>print(...)</code> — the two totals side by side.</li><li><code>print(...)</code> — the same comparison as a single number.</li><li><code>print("big client error, FedAvg"...)</code> — error on the 700-example client.</li><li><code>print("big client error, equal"...)</code> — the same client under equal weights.</li><li><code>idx = ...</code> — x positions shared by both bar groups.</li><li><code>plt.figure(...)</code> — a fresh canvas.</li><li><code>plt.bar(idx - 0.2, ...)</code> — green bars, shifted left.</li><li><code>plt.bar(idx + 0.2, ...)</code> — red bars, shifted right.</li><li><code>plt.xticks(...)</code> — replace positions with client names.</li><li><code>plt.ylabel(...)</code> — names the vertical axis.</li><li><code>plt.legend()</code> — a key for green versus red.</li><li><code>plt.title(...)</code> — the takeaway.</li><li><code>plt.show()</code> — draws the chart.</li></ul>

</details>

## Your turn

Change `sizes` in section 1 to `[400, 300, 300]` and predict the FedAvg value before rerunning. Then drag section 4 to five local epochs: does the endpoint change, or only the speed?